# Lesson 10A: Continuous Control - Theory

## Introduction

Robotics and physical systems require continuous action spaces. **Deterministic policy gradients (DPG)** and entropy-regularized RL enable learning smooth, continuous policies.

- **DDPG**: Deep Deterministic Policy Gradient (off-policy continuous control)
- **TD3**: Twin Delayed DDPG (addresses overestimation in DDPG)
- **SAC**: Soft Actor-Critic (entropy-regularized, state-of-the-art)

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## Part 1: Deterministic Policy Gradient Theorem

### From Stochastic to Deterministic

Policy gradient:
$$\nabla J = \mathbb{E}[\nabla_a Q(s,a) \nabla_\theta \mu_\theta(s)]$$

Where:
- **μ(s)**: deterministic policy (returns single action, not distribution)
- **Q(s,a)**: action-value function
- No need to marginalize over actions: ∇_a cancels the stochasticity

**Advantage**: Off-policy learning becomes feasible with deterministic policies.

## Part 2: DDPG (Deep Deterministic Policy Gradient)

### Actor-Critic with Experience Replay

- **Actor (μ_θ)**: deterministic policy → continuous action
- **Critic (Q_w)**: action-value function
- **Replay buffer**: off-policy experience
- **Target networks**: stabilize learning (Q_target, μ_target)

### Update Rules

Critic loss:
$$L = (Q_w(s,a) - [r + \gamma Q_{w'}(s', \mu_{\theta'}(s'))])^2$$

Actor loss:
$$L = -\mathbb{E}[Q_w(s, \mu_\theta(s))]$$

Add exploration noise to actions during training.

In [ ]:
# Conceptual DDPG pseudocode
def ddpg_update(replay_buffer, actor, critic, actor_target, critic_target, gamma=0.99):
    """
    One gradient step for both actor and critic.
    """
    # Sample from replay buffer
    states, actions, rewards, next_states, dones = replay_buffer.sample(batch_size=64)
    
    # Compute TD target
    next_actions = actor_target(next_states)  # deterministic
    target_q = rewards + gamma * (1 - dones) * critic_target(next_states, next_actions)
    
    # Critic update: minimize Bellman error
    q_values = critic(states, actions)
    critic_loss = mse_loss(q_values, target_q.detach())
    critic.optimize(critic_loss)
    
    # Actor update: maximize Q (policy gradient)
    policy_actions = actor(states)
    actor_loss = -critic(states, policy_actions).mean()
    actor.optimize(actor_loss)
    
    # Soft update target networks
    soft_update(actor_target, actor, tau=0.001)
    soft_update(critic_target, critic, tau=0.001)

print("DDPG: defined conceptually")

## Part 3: TD3 (Twin Delayed DDPG)

### Addressing Overestimation Bias

DDPG suffers from **Q-value overestimation**: max over continuous actions inflates estimates.

**TD3 fixes**:
1. **Twin critics**: two Q-networks, take minimum → pessimistic estimate
2. **Delayed updates**: update actor less frequently (every N critic steps)
3. **Target policy smoothing**: add noise to target actions (reduce exploitability)

$$y = r + \gamma \min_{i=1,2} Q_{w_i'}(s', \tilde{\mu}(s'))$$

Where $\tilde{\mu}(s') = \mu_{\theta'}(s') + \epsilon$, $\epsilon \sim \text{Clip}(\mathcal{N}(0,\sigma), -c, c)$

## Part 4: SAC (Soft Actor-Critic)

### Maximum Entropy RL

Trade off return vs. policy entropy:
$$J = \mathbb{E}[r(s,a) + \alpha H(\pi(\cdot|s))]$$

Where **α** is the temperature (entropy coefficient).

**Advantages**:
- Automatic exploration: higher entropy → more exploration
- Robust to reward misspecification
- Learns stochastic policies (can represent multimodal behaviors)

### Implementation

- Actor: stochastic policy (Gaussian)
- Two critics (like TD3)
- Automatic temperature tuning (learns α)

In [ ]:
# SAC update loop (pseudocode)
def sac_update(replay_buffer, actor, critic1, critic2, alpha=0.2, gamma=0.99):
    """
    Soft Actor-Critic update.
    """
    states, actions, rewards, next_states, dones = replay_buffer.sample()
    
    # Actor update: maximize Q - alpha * H(pi)
    next_actions, next_log_probs = actor(next_states)  # stochastic
    target_q = torch.min(
        critic1(next_states, next_actions),
        critic2(next_states, next_actions)
    )
    target = rewards + gamma * (1 - dones) * (target_q - alpha * next_log_probs)
    
    # Critic update: Bellman regression
    q1 = critic1(states, actions)
    q2 = critic2(states, actions)
    critic_loss = mse_loss(q1, target) + mse_loss(q2, target)
    
    # Actor update: policy gradient with entropy
    actions, log_probs = actor(states)
    q_min = torch.min(critic1(states, actions), critic2(states, actions))
    actor_loss = (alpha * log_probs - q_min).mean()

print("SAC: defined conceptually")

## Part 5: Comparison

| Algorithm | Policy | Exploration | Convergence | Complexity |
|-----------|--------|-------------|-------------|------------|
| DDPG | Deterministic | Action noise | Moderate | Medium |
| TD3 | Deterministic | Action noise + smoothing | Good | Medium-High |
| SAC | Stochastic | Entropy | Stable | High |

**Recommendation**:
- Start with **PPO** for most tasks (easier to tune)
- Use **TD3** for robotics if sample efficiency is critical
- Use **SAC** for maximum robustness and multi-task learning

## Key Concepts

1. **Deterministic policy gradients**: ∇_a factors out, enabling off-policy learning
2. **DDPG**: Actor-Critic + replay buffer + target networks
3. **TD3**: Twin critics + delayed updates + target noise
4. **SAC**: Entropy-regularized, automatic temperature tuning, state-of-the-art

Next: Implement TD3 and SAC in PyTorch on robotic control tasks.